In [1]:
from google.colab import userdata
import os

os.environ['KAGGLE_API_TOKEN']=userdata.get('KAGGLE_API_TOKEN')

In [2]:
!kaggle competitions download -c tensorflow-great-barrier-reef -p /content/data
!unzip -q /content/data/tensorflow-great-barrier-reef.zip -d /content/data

100% 14.2G/14.2G [11:25<00:00, 22.2MB/s]



In [4]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.7 MB/s eta 0:00:00


In [5]:
import os
import shutil
from ultralytics import YOLO
import dataset


def clean_yolo_dataset():
    """
    Remove the previously generated YOLO dataset so that
    the baseline model starts from a clean dataset.
    """
    dirs_to_clean = [
        'data/images/train',
        'data/images/val',
        'data/labels/train',
        'data/labels/val'
    ]

    for directory in dirs_to_clean:
        if os.path.exists(directory):
            shutil.rmtree(directory)

        os.makedirs(directory, exist_ok=True)


def train_baseline(
    ratio=2.0,
    epochs=20,
    imgsz=320,
    batch=32
):
    print("\n==========================================")
    print("          BASELINE MODEL")
    print(f"     Negative Ratio: {ratio}")
    print("==========================================\n")

    # 0. Clean previously generated YOLO dataset
    clean_yolo_dataset()

    # 1. Load data and apply the fixed train/validation split
    full_df = dataset.load_data(
        'data/train.csv',
        'data/splits.csv'
    )

    train_df = full_df[
        full_df['split'] == 'train'
    ].copy()

    val_df = full_df[
        full_df['split'] == 'val'
    ].copy()

    # 2. Apply chosen negative ratio ONLY to training data
    train_df = dataset.filter_negatives(
        train_df,
        ratio=ratio
    )

    print(f"Training frames:   {len(train_df)}")
    print(f"Validation frames: {len(val_df)}")

    # 3. Generate YOLO dataset
    dataset.write_yolo_labels(
        train_df,
        labels_dir='data/labels/train'
    )

    dataset.write_yolo_labels(
        val_df,
        labels_dir='data/labels/val'
    )

    dataset.link_images(
        train_df,
        'data/train_images',
        images_dir='data/images/train'
    )

    dataset.link_images(
        val_df,
        'data/train_images',
        images_dir='data/images/val'
    )

    dataset.write_data_yaml(
        'data/data.yaml',
        'data/images/train',
        'data/images/val'
    )

    # 4. Initialize pretrained YOLOv8n
    model = YOLO('yolov8n.pt')

    # 5. Train baseline model
    model.train(
        data='data/data.yaml',
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        workers=4,
        cache='ram',
        plots=True,
        save=True,
        seed=42,
        project='baseline_model',
        name='yolov8n_ratio_2',
        exist_ok=True
    )

    # 6. Evaluate baseline model
    metrics = model.val()

    print("\n=============== BASELINE RESULTS ===============")
    print(f"mAP50:    {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")

    return metrics


if __name__ == '__main__':
    train_baseline()

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

          BASELINE MODEL
     Negative Ratio: 2.0

Training frames:   12846
Validation frames: 4057
Ultralytics 8.4.123 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=N